## Convert Mat -> png

In [2]:
%pip install mat73

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import numpy as np
import scipy.io
import mat73
import cv2

# === Configuration ===
INPUT_ROOT_DIR = 'Figshare'
OUTPUT_DIR = 'Processed_Images'

# 1: Meningioma, 2: Glioma, 3: Pituitary tumor
LABEL_MAP = {
    1: 'Meningioma',
    2: 'Glioma',
    3: 'Pituitary'
}
# =====================

def create_directory(path):
    if not os.path.exists(path):
        os.makedirs(path)

def normalize_image(img_data):
    """Normalize MRI data to 0-255 8-bit image."""
    # Handle cases where image might be flat or have extra dims
    if img_data.ndim == 3:
        img_data = img_data.squeeze() # Remove extra dimensions like (512, 512, 1)
        
    img_normalized = (img_data - np.min(img_data)) / (np.max(img_data) - np.min(img_data))
    img_uint8 = (img_normalized * 255).astype(np.uint8)
    return img_uint8

def get_clean_label(raw_label):
    """
    Forcefully extracts a standard Python int from various numpy/matlab formats.
    Handles: scalar, array([1]), array([[1]]), array(1.0), etc.
    """
    try:
        # If it's a numpy object/array, try to get the scalar item
        if isinstance(raw_label, (np.ndarray, np.generic)):
            val = raw_label.item()
        else:
            val = raw_label
            
        return int(val)
    except Exception:
        # Fallback for complex nested structures often found in older scipy versions
        try:
            return int(raw_label[0][0])
        except:
            return int(raw_label.flatten()[0])

def process_dataset():
    create_directory(OUTPUT_DIR)
    for label_name in LABEL_MAP.values():
        create_directory(os.path.join(OUTPUT_DIR, label_name))

    count = 0
    errors = 0

    print("Scanning directories...")
    
    for root, dirs, files in os.walk(INPUT_ROOT_DIR):
        for file in files:
            if file.endswith('.mat') and not file.startswith('cvind'):
                mat_path = os.path.join(root, file)
                
                try:
                    image_matrix = None
                    raw_label = None
                    
                    # --- STRATEGY 1: Try Scipy (Old format) ---
                    try:
                        mat_contents = scipy.io.loadmat(mat_path)
                        cjdata = mat_contents['cjdata']
                        # Scipy structure extraction usually requires [0][0]
                        image_matrix = cjdata['image'][0][0] 
                        raw_label = cjdata['label'][0][0]
                        
                    # --- STRATEGY 2: Try Mat73 (New format v7.3) ---
                    except NotImplementedError:
                        mat_contents = mat73.loadmat(mat_path)
                        cjdata = mat_contents['cjdata']
                        image_matrix = cjdata['image']
                        raw_label = cjdata['label']

                    if image_matrix is None or raw_label is None:
                        raise ValueError("Data extraction failed")

                    # --- CRITICAL FIX: Sanitize Label ---
                    # Convert whatever array structure we got into a simple Integer
                    label = get_clean_label(raw_label)

                    # Check if label is valid (must be 1, 2, or 3)
                    if label not in LABEL_MAP:
                        # Sometimes files have garbage labels, skip them or log them
                        # But for this dataset, it should be 1-3
                        print(f"Warning: File {file} has unexpected label {label}")
                        continue

                    class_name = LABEL_MAP[label]
                    processed_img = normalize_image(image_matrix)

                    # Save File
                    base_filename = os.path.splitext(file)[0]
                    output_filename = f"{base_filename}.png"
                    output_path = os.path.join(OUTPUT_DIR, class_name, output_filename)

                    cv2.imwrite(output_path, processed_img)
                    
                    count += 1
                    if count % 100 == 0:
                        print(f"Processed {count} images... (Current: {file})")

                except Exception as e:
                    # Print full error only for the first few to avoid spamming
                    if errors < 5:
                        print(f"FAILED on {file}: {e}")
                    elif errors == 5:
                        print("Too many errors displaying, suppressing further error logs...")
                    errors += 1

    print("\nConversion Finished!")
    print(f"Successfully converted: {count} images.")
    print(f"Errors: {errors}")

if __name__ == "__main__":
    if not os.path.exists(INPUT_ROOT_DIR):
        print(f"Error: Folder '{INPUT_ROOT_DIR}' not found.")
    else:
        process_dataset()

Scanning directories...
Processed 100 images... (Current: 189.mat)
Processed 200 images... (Current: 279.mat)
Processed 300 images... (Current: 369.mat)
Processed 400 images... (Current: 459.mat)
Processed 500 images... (Current: 549.mat)
Processed 600 images... (Current: 639.mat)
Processed 700 images... (Current: 729.mat)
Processed 800 images... (Current: 1566.mat)
Processed 900 images... (Current: 1666.mat)
Processed 1000 images... (Current: 1766.mat)
Processed 1100 images... (Current: 1866.mat)
Processed 1200 images... (Current: 1966.mat)
Processed 1300 images... (Current: 2066.mat)
Processed 1400 images... (Current: 2166.mat)
Processed 1500 images... (Current: 2266.mat)
Processed 1600 images... (Current: 2366.mat)
Processed 1700 images... (Current: 2466.mat)
Processed 1800 images... (Current: 2566.mat)
Processed 1900 images... (Current: 2666.mat)
Processed 2000 images... (Current: 2766.mat)
Processed 2100 images... (Current: 2866.mat)
Processed 2200 images... (Current: 2966.mat)
Pr

In [6]:
import os
import shutil
from tqdm import tqdm

# ================= CONFIGURATION =================
# 1. The folder that contains 'Training' and 'Testing' folders
SOURCE_DIR = 'BTMK' 

# 2. The new folder where you want EVERYTHING to go
DEST_DIR = 'BTMK_Merged'
# =================================================

def merge_folders():
    # Check if source exists
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: Could not find source folder '{SOURCE_DIR}'")
        return

    # Create the destination root
    if not os.path.exists(DEST_DIR):
        os.makedirs(DEST_DIR)

    # We assume the structure is Source -> [Training, Testing] -> [Classname]
    subsets = ['Training', 'Testing']
    
    total_moved = 0

    for subset in subsets:
        subset_path = os.path.join(SOURCE_DIR, subset)
        
        # Check if Training/Testing folder exists
        if not os.path.exists(subset_path):
            print(f"Warning: '{subset}' folder not found in {SOURCE_DIR}")
            continue

        # Get list of classes (Glioma, Meningioma, etc.)
        classes = [d for d in os.listdir(subset_path) if os.path.isdir(os.path.join(subset_path, d))]

        for class_name in classes:
            # Create the class folder in the Destination
            dest_class_path = os.path.join(DEST_DIR, class_name)
            if not os.path.exists(dest_class_path):
                os.makedirs(dest_class_path)

            # Get files in the source class folder
            src_class_path = os.path.join(subset_path, class_name)
            files = os.listdir(src_class_path)

            print(f"Processing {subset} - {class_name}...")
            
            for filename in tqdm(files):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif')):
                    
                    # Construct full source path
                    src_file = os.path.join(src_class_path, filename)
                    
                    # Construct new filename to avoid duplicates
                    # Example: glioma_tr_image1.jpg
                    prefix = "tr" if subset == 'Training' else "ts"
                    new_filename = f"{class_name}_{prefix}_{filename}"
                    
                    dest_file = os.path.join(dest_class_path, new_filename)
                    
                    # Copy the file
                    shutil.copy2(src_file, dest_file)
                    total_moved += 1

    print("-" * 30)
    print(f"✅ Merge Complete!")
    print(f"Total images merged: {total_moved}")
    print(f"All files are now in: {os.path.abspath(DEST_DIR)}")

if __name__ == "__main__":
    merge_folders()

Processing Training - glioma...


100%|██████████| 1321/1321 [00:00<00:00, 1814.11it/s]


Processing Training - meningioma...


100%|██████████| 1339/1339 [00:00<00:00, 1814.06it/s]


Processing Training - notumor...


...

Processing Testing - notumor...


100%|██████████| 405/405 [00:00<00:00, 1823.50it/s]


Processing Testing - pituitary...


100%|██████████| 300/300 [00:00<00:00, 1521.57it/s]

------------------------------
✅ Merge Complete!
Total images merged: 7023
All files are now in: a:\MRI\BTMK_Merged


In [7]:
import os
import shutil
import random
import math
from tqdm import tqdm

# ================= CONFIGURATION =================
# 1. The folder containing the merged classes (from previous step)
SOURCE_DIR = 'BTMK_Merged'

# 2. The output folder name
OUTPUT_DIR = 'BTMK'

# 3. Split Ratios (Must add up to 1.0)
TRAIN_RATIO = 0.70  # 70%
VAL_RATIO   = 0.15  # 15%
TEST_RATIO  = 0.15  # 15%
# =================================================

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def split_dataset():
    # check if source exists
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: Source folder '{SOURCE_DIR}' not found.")
        return

    # Get list of classes (subfolders in source)
    classes = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]
    
    print(f"Found classes: {classes}")
    print(f"Splitting into: Train ({TRAIN_RATIO}), Val ({VAL_RATIO}), Test ({TEST_RATIO})")

    for class_name in classes:
        class_dir = os.path.join(SOURCE_DIR, class_name)
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        # 1. Shuffle images to ensure random split
        random.shuffle(images)
        
        # 2. Calculate split counts
        total_images = len(images)
        train_count = math.floor(total_images * TRAIN_RATIO)
        val_count = math.floor(total_images * VAL_RATIO)
        # Test gets the remainder to avoid rounding errors
        test_count = total_images - train_count - val_count

        # 3. Slice lists
        train_imgs = images[:train_count]
        val_imgs = images[train_count : train_count + val_count]
        test_imgs = images[train_count + val_count :]

        # 4. Copy files
        splits = [
            ('train', train_imgs),
            ('val', val_imgs),
            ('test', test_imgs)
        ]

        print(f"\nProcessing '{class_name}': {total_images} images")
        
        for split_name, img_list in splits:
            # Create destination folder: Final_Dataset_Split/train/glioma
            dest_path = os.path.join(OUTPUT_DIR, split_name, class_name)
            create_dir(dest_path)
            
            print(f"  -> Copying {len(img_list)} images to {split_name}...")
            
            for img_name in tqdm(img_list, leave=False):
                src_file = os.path.join(class_dir, img_name)
                dst_file = os.path.join(dest_path, img_name)
                shutil.copy2(src_file, dst_file)

    print("\n" + "="*30)
    print("✅ Split Complete!")
    print(f"Dataset location: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    split_dataset()

Found classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Splitting into: Train (0.7), Val (0.15), Test (0.15)

Processing 'glioma': 1621 images
  -> Copying 1134 images to train...


  -> Copying 243 images to val...


  -> Copying 244 images to test...



Processing 'meningioma': 1645 images
  -> Copying 1151 images to train...


  -> Copying 246 images to val...


...

  -> Copying 300 images to test...



Processing 'pituitary': 1757 images
  -> Copying 1229 images to train...


  -> Copying 263 images to val...


  -> Copying 265 images to test...



✅ Split Complete!
Dataset location: a:\MRI\BTMK


In [8]:
import os
import shutil
import random
import math
from tqdm import tqdm

# ================= CONFIGURATION =================
# 1. The folder containing the merged classes (from previous step)
SOURCE_DIR = 'Figshare'

# 2. The output folder name
OUTPUT_DIR = 'Figsplit'

# 3. Split Ratios (Must add up to 1.0)
TRAIN_RATIO = 0.70  # 70%
VAL_RATIO   = 0.15  # 15%
TEST_RATIO  = 0.15  # 15%
# =================================================

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def split_dataset():
    # check if source exists
    if not os.path.exists(SOURCE_DIR):
        print(f"Error: Source folder '{SOURCE_DIR}' not found.")
        return

    # Get list of classes (subfolders in source)
    classes = [d for d in os.listdir(SOURCE_DIR) if os.path.isdir(os.path.join(SOURCE_DIR, d))]
    
    print(f"Found classes: {classes}")
    print(f"Splitting into: Train ({TRAIN_RATIO}), Val ({VAL_RATIO}), Test ({TEST_RATIO})")

    for class_name in classes:
        class_dir = os.path.join(SOURCE_DIR, class_name)
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        
        # 1. Shuffle images to ensure random split
        random.shuffle(images)
        
        # 2. Calculate split counts
        total_images = len(images)
        train_count = math.floor(total_images * TRAIN_RATIO)
        val_count = math.floor(total_images * VAL_RATIO)
        # Test gets the remainder to avoid rounding errors
        test_count = total_images - train_count - val_count

        # 3. Slice lists
        train_imgs = images[:train_count]
        val_imgs = images[train_count : train_count + val_count]
        test_imgs = images[train_count + val_count :]

        # 4. Copy files
        splits = [
            ('train', train_imgs),
            ('val', val_imgs),
            ('test', test_imgs)
        ]

        print(f"\nProcessing '{class_name}': {total_images} images")
        
        for split_name, img_list in splits:
            # Create destination folder: Final_Dataset_Split/train/glioma
            dest_path = os.path.join(OUTPUT_DIR, split_name, class_name)
            create_dir(dest_path)
            
            print(f"  -> Copying {len(img_list)} images to {split_name}...")
            
            for img_name in tqdm(img_list, leave=False):
                src_file = os.path.join(class_dir, img_name)
                dst_file = os.path.join(dest_path, img_name)
                shutil.copy2(src_file, dst_file)

    print("\n" + "="*30)
    print("✅ Split Complete!")
    print(f"Dataset location: {os.path.abspath(OUTPUT_DIR)}")

if __name__ == "__main__":
    split_dataset()

Found classes: ['Glioma', 'Meningioma', 'Pituitary']
Splitting into: Train (0.7), Val (0.15), Test (0.15)

Processing 'Glioma': 1426 images
  -> Copying 998 images to train...


  -> Copying 213 images to val...


  -> Copying 215 images to test...



Processing 'Meningioma': 708 images
  -> Copying 495 images to train...


  -> Copying 106 images to val...


  -> Copying 107 images to test...



Processing 'Pituitary': 930 images
  -> Copying 651 images to train...


  -> Copying 139 images to val...


  -> Copying 140 images to test...



✅ Split Complete!
Dataset location: a:\MRI\Figsplit
